In [2]:
# this is to check if the labels I am using are correct
import os

In [3]:
import json 

def add_fold(fold, case = None): 
    if case: 
        case.append({'train': fold['train'], 'val': fold['val']})
    else: 
        case = [{'train': fold['train'], 'val': fold['val']}]
        return case

# SplitDirectory
SPLIT_DIR = f"{os.environ['nnUNet_preprocessed']}/{{dataset}}/splits_final.json"

with open(SPLIT_DIR.format(dataset='Dataset999_AutoPet'), 'r') as f:
    splits_for_all_files = json.load(f)

with open(SPLIT_DIR.format(dataset='Dataset111_AutoPet'), 'r') as f:
    splits_for_AL = json.load(f)

SPLIT_TO_REPLICATE = 4

print(len(splits_for_AL[SPLIT_TO_REPLICATE]['train']))
print(len(splits_for_all_files[11]['train']))

first_X_percent = add_fold(splits_for_all_files[SPLIT_TO_REPLICATE + 2])
# original 10%
print(len(first_X_percent[0]['train']))

312
1043
312


In [4]:
fdg_in_X_percent = [file for file in splits_for_AL[SPLIT_TO_REPLICATE]['train'] if 'fdg' in file]
psma_in_X_percent = [file for file in splits_for_AL[SPLIT_TO_REPLICATE]['train'] if 'psma' in file]

print(f"Number of fdg files in fold {SPLIT_TO_REPLICATE}:", len(fdg_in_X_percent))
print(f"Number of psma files in fold {SPLIT_TO_REPLICATE}:", len(psma_in_X_percent))


Number of fdg files in fold 4: 199
Number of psma files in fold 4: 113


In [5]:
original_10_percent = splits_for_AL[0]
original_psma_files = [file for file in original_10_percent['train'] if 'psma' in file]
original_fdg_files = [file for file in original_10_percent['train'] if 'fdg' in file]
print("Number of fdg files in original 10%:", len(original_fdg_files))
print("Number of psma files in original 10%:", len(original_psma_files))

print(f"We will now randomize {len(fdg_in_X_percent) - len(original_fdg_files)} fdg files and {len(psma_in_X_percent) - len(original_psma_files)} psma files to create.")

# now to select a random 30 % of the data
all_psma_files = [file for file in splits_for_all_files[11]['train'] if 'psma' in file]
all_fdg_files = [file for file in splits_for_all_files[11]['train'] if 'fdg' in file]

all_psma_files = [file for file in all_psma_files if file not in original_10_percent['train']]
all_fdg_files = [file for file in all_fdg_files if file not in original_10_percent['train']]

# set seed
import random

cases = first_X_percent.copy()  # start with the original 40% case

for seed in [42, 43, 44, 45]:
    random.seed(seed)
    randomized_psma_files = random.sample(all_psma_files, len(psma_in_X_percent) - len(original_psma_files))
    randomized_fdg_files = random.sample(all_fdg_files, len(fdg_in_X_percent) - len(original_fdg_files))

    new_random_psma_files = original_psma_files + randomized_psma_files
    new_random_fdg_files = original_fdg_files + randomized_fdg_files

    new_case = {'train': new_random_psma_files + new_random_fdg_files, 'val': splits_for_AL[SPLIT_TO_REPLICATE]['val']}  

    add_fold(new_case, case=cases)


Number of fdg files in original 10%: 66
Number of psma files in original 10%: 37
We will now randomize 133 fdg files and 76 psma files to create.


In [6]:
for case in cases: 
    print(len(case['train']), len(case['val']))

312 247
312 247
312 247
312 247
312 247


In [7]:
# check the overlap between the 5 different cases in similarly matrix using set

cases_as_sets = [set(case['train']) for case in cases]

overlap_matrix = [[len(set1.intersection(set2)) for set2 in cases_as_sets] for set1 in cases_as_sets]
print("Overlap matrix between the 5 cases:")
for i in range(len(overlap_matrix)):
    print(f"Case {i+1}: {overlap_matrix[i]}")

print("Intersection of all 5 cases:", len(set.intersection(*cases_as_sets)))


Overlap matrix between the 5 cases:
Case 1: [312, 153, 157, 150, 155]
Case 2: [153, 312, 153, 160, 150]
Case 3: [157, 153, 312, 141, 140]
Case 4: [150, 160, 141, 312, 147]
Case 5: [155, 150, 140, 147, 312]
Intersection of all 5 cases: 104


In [8]:
# intersection with GREEDY 
greedy = splits_for_AL[SPLIT_TO_REPLICATE]
# get train intersection for greedy 
greedy_train_set = set(greedy['train'])

print(len(greedy_train_set))

for i, case in enumerate(cases):
    case_train_set = set(case['train'])
    intersection_with_greedy = len(greedy_train_set.intersection(case_train_set))
    print(f"Intersection of Case {i+1} with GREEDY: {intersection_with_greedy}")
    

312
Intersection of Case 1 with GREEDY: 157
Intersection of Case 2 with GREEDY: 153
Intersection of Case 3 with GREEDY: 145
Intersection of Case 4 with GREEDY: 152
Intersection of Case 5 with GREEDY: 154


In [9]:
# export cases as json 
with open('randomized_cases_30_percent.json', 'w') as f:
    json.dump(cases, f, indent=4)

Overlap matrix between the 5 cases:
Case 1: [417, 206, 200, 206, 208]
Case 2: [206, 417, 203, 210, 205]
Case 3: [200, 203, 417, 206, 223]
Case 4: [206, 210, 206, 417, 203]
Case 5: [208, 205, 223, 203, 417]
Intersection of all 5 cases: 105
Union of all 5 cases: 913


In [10]:
print(len(randomized_psma_files) + len(original_psma_files))
print(len(randomized_fdg_files) + len(original_fdg_files))



113
199


In [23]:
139 + 245

384

In [12]:
len(randomized_40_percent[0]['train'])

417